In [33]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [34]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [35]:
import pandas as pd
import numpy as np

In [36]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [37]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [38]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [39]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [40]:
# no letters in common
w1b & w2b

0

In [41]:
# letters in common
w1b & w3b

147456

In [42]:
# bitwise or
w1b | w2b

673975

In [43]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

In [44]:
byte_encode_words(ascii_lowercase)

67108863

# BUILD LEVEL 2

In [45]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [46]:
l2_df['l2'].unique().shape

(640023,)

# BUILD LEVELS 3 THROUGH 5

In [47]:
l2_df.shape

(3213696, 3)

In [48]:
l2_all = l2_list[:, 2]

In [49]:
l2_all.shape

(3213696,)

In [50]:
l2_df_test = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [51]:
l2_all = l2_df_test['l2'].to_numpy(dtype=np.int32)

In [86]:
# so, now, let's try computing all possible pairs
start_pos = 0
total_output = np.full(shape = (100000000, 7), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df_test.iloc[:20000].iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # compare the current l2 to all l2 - this will find all instances
    # indexer for l2 and l3
    positional_idx_l3 = (l2_all & l2) == 0

    # combined l2 words and words with different letters
    output_array_w3b = l2_all[positional_idx_l3]    
    
    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    # create the temp output
    n_rows = output_array_l3.shape[0]
    if n_rows > 0:
        #print(n_rows)
        temp_output = np.zeros(shape = (n_rows, 7), dtype = np.int32)
        temp_output[:, 0] = w1b
        temp_output[:, 1] = w2b
        temp_output[:, 2] = l2
        temp_output[:, 3] = output_array_w3b
        temp_output[:, 4] = output_array_l3

        test_output = l2_df_test.loc[positional_idx_l3, ['w1b', 'w2b']].to_numpy(dtype = np.int32)

        print(test_output.shape)
        #print(temp_output.shape)
        print(test_output[:, 0].shape)
        print(test_output[:, 1].shape)
        print(temp_output[:, 5].shape)
        temp_output[:, 5] = test_output[:, 0]
        temp_output[:, 6] = test_output[:, 1]

        # gather it all
        total_output[start_pos:start_pos + n_rows, :] = temp_output
        start_pos += n_rows
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)

    row_index += 1

0 0
(1, 2)
(1,)
(1,)
(1,)
(5, 2)
(5,)
(5,)
(5,)
(7, 2)
(7,)
(7,)
(7,)
(1, 2)
(1,)
(1,)
(1,)
(8, 2)
(8,)
(8,)
(8,)
(12, 2)
(12,)
(12,)
(12,)
(1, 2)
(1,)
(1,)
(1,)
(4, 2)
(4,)
(4,)
(4,)
(26, 2)
(26,)
(26,)
(26,)
(60, 2)
(60,)
(60,)
(60,)
(37, 2)
(37,)
(37,)
(37,)
(17, 2)
(17,)
(17,)
(17,)
(8, 2)
(8,)
(8,)
(8,)
(71, 2)
(71,)
(71,)
(71,)
(48, 2)
(48,)
(48,)
(48,)
(50, 2)
(50,)
(50,)
(50,)
(75, 2)
(75,)
(75,)
(75,)
(64, 2)
(64,)
(64,)
(64,)
(20, 2)
(20,)
(20,)
(20,)
(29, 2)
(29,)
(29,)
(29,)
(31, 2)
(31,)
(31,)
(31,)
(35, 2)
(35,)
(35,)
(35,)
(16, 2)
(16,)
(16,)
(16,)
(35, 2)
(35,)
(35,)
(35,)
(27, 2)
(27,)
(27,)
(27,)
(18, 2)
(18,)
(18,)
(18,)
(157, 2)
(157,)
(157,)
(157,)
(59, 2)
(59,)
(59,)
(59,)
(11, 2)
(11,)
(11,)
(11,)
(40, 2)
(40,)
(40,)
(40,)
(94, 2)
(94,)
(94,)
(94,)
(77, 2)
(77,)
(77,)
(77,)
(35, 2)
(35,)
(35,)
(35,)
(17, 2)
(17,)
(17,)
(17,)
(18, 2)
(18,)
(18,)
(18,)
(1, 2)
(1,)
(1,)
(1,)
(2, 2)
(2,)
(2,)
(2,)
(2, 2)
(2,)
(2,)
(2,)
(8, 2)
(8,)
(8,)
(8,)
(24, 2)
(24,)
(24,)
(24,)


In [87]:
start_pos

227493

In [89]:
testo = total_output[:start_pos]

In [90]:
testo.shape

(227493, 7)

In [91]:
testo

array([[   20491,   532756,   553247, ..., 18808319,  1313824, 16941248],
       [   20491,   788500,   808991, ..., 18808319, 17835040,   164288],
       [   20491,   788500,   808991, ..., 56523263, 51511328,  4202944],
       ...,
       [  262293,  1087552,  1349845, ..., 56557045, 34210080, 20997120],
       [  262293,  1087552,  1349845, ..., 25099765, 21119008,  2630912],
       [  262293,  1087552,  1349845, ..., 64944605, 25837568, 37757192]],
      shape=(227493, 7), dtype=int32)

In [ ]:
# save to disk
np.save('H:/project/wordle_fun/l2l3.npy', arr = testo)

In [ ]:
# load data
total_output = np.load(file = 'H:/project/wordle_fun/l2l3.npy')


In [ ]:
total_output.shape

In [ ]:
total_output

In [59]:
testo

array([[   20491,   532756,   553247, 18255072, 18808319],
       [   20491,   788500,   808991, 17999328, 18808319],
       [   20491,   788500,   808991, 55714272, 56523263],
       ...,
       [  262293,  1087552,  1349845, 55207200, 56557045],
       [  262293,  1087552,  1349845, 23749920, 25099765],
       [  262293,  1087552,  1349845, 63594760, 64944605]],
      shape=(227493, 5), dtype=int32)

In [60]:
20491 | 532756

553247

In [61]:
553247 | 18255072

18808319

In [62]:
l2_df.loc[l2_df['l2'] == 18255072, ]

,w1b,w2b,l2
2058913,1313824,16941248,18255072


In [ ]:
20491,   532756,   553247, 18255072, 18808319],

In [ ]:
20491 | 532756 | 

In [ ]:
temp_output[:, 0] = w1b
temp_output[:, 1] = w2b
temp_output[:, 2] = l2
temp_output[:, 3] = output_array_l3

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
